# Phase 2 — Interpretability: sensor and time attributions (14)

Issue #73. What do the sequence models use to predict remaining useful life (RUL), and does it agree with what the Phase 1 tree models used? Everything here explains the frozen models from #71 (checkpoints in `results/models/final/`, gitignored) on the 100 official FD001 test engines, at each engine's last recorded cycle, the cycle that was scored. Nothing is trained again, and the attributions do not change any test score.

**Terms**

- **Integrated gradients (IG):** for one input window, IG moves the input in a straight line from a *baseline* to the real window and adds up the model's gradients along the way. Each sensor at each time step gets a share of the prediction, in cycles of RUL. The shares add up to the prediction minus the baseline's prediction (*completeness*), which section 1 checks. The baseline here is the all-zero window: every sensor at its training mean, since the inputs are z-scored on the training engines.
- **Window:** the 30 most recent cycles the model sees. Position 0 is the last (scored) cycle, and −29 is the oldest.
- **SHAP:** the Phase 1 method for tree models; mean absolute SHAP value is a feature's average contribution, in cycles.
- **Attention:** the Transformer's weights for how much each cycle in the window draws on every other cycle.
- **Gate:** Hybrid A mixes its two branches as `g * sequence + (1 - g) * features`. A gate near 1 means it leans on the sensor windows, near 0 on the engineered features.

**Which model.** The best sequence model is the Transformer: lowest cross-validated selection RMSE among the sequence models (#60, the choice #72 also used). The LSTM and temporal CNN get the same IG analysis as a check on whether the sensor ranking depends on the architecture.

In [1]:
from pathlib import Path
import json
import re
import sys

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import spearmanr

from rul.data.labels import add_test_rul
from rul.data.load import load_subset
from rul.training.runs import TrainedModel

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "scripts"))
MODELS_DIR = ROOT / "results/models/final"
METRICS = ROOT / "results/metrics"
FIGURES = ROOT / "results/figures"
SEEDS = (42, 43, 44, 45, 46)
STEPS = 256  # integration steps for IG
STAGES = {"late": (0, 50), "mid": (50, 100), "early": (100, np.inf)}
torch.manual_seed(0)

train, test, rul_test = load_subset("FD001", data_dir=ROOT / "data/raw/CMAPSSData")
labelled = add_test_rul(test, rul_test)
assert labelled.index.equals(test.index)
last_rows = labelled.sort_values(["unit", "cycle"]).groupby("unit").tail(1).sort_values("unit")
y = last_rows["rul"].to_numpy(dtype=float)
stage_of = pd.cut(y, [lo for lo, _ in STAGES.values()] + [np.inf], labels=list(STAGES)).astype(str)
final = pd.read_csv(METRICS / "fd001_final_test_predictions.csv")


def load(model, seed):
    return TrainedModel.load(MODELS_DIR / f"fd001_{model}_seed{seed}.pt")


def last_windows(trained):
    """Input tensors for each test engine's last cycle, in unit order."""
    batch = trained.inputs.transform(test, target=None, stride=1)
    pos = pd.Series(np.arange(len(batch.index)), index=batch.index["row"].to_numpy()).loc[last_rows.index]
    return [torch.as_tensor(a[pos.to_numpy()]) for a in batch.inputs]


def integrated_gradients(net, x, steps=STEPS):
    """IG of ``net(x)`` (one output per window) from a zero baseline, midpoint rule."""
    net.eval()
    total = torch.zeros_like(x)
    for a in (torch.arange(steps, dtype=x.dtype) + 0.5) / steps:
        xi = (a * x).requires_grad_(True)
        (grad,) = torch.autograd.grad(net(xi).sum(), xi)
        total += grad
    return x * total / steps


print(f"{len(y)} test engines: " + ", ".join(f"{s} {int((stage_of == s).sum())}" for s in STAGES))

100 test engines: late 33, mid 34, early 33


## 1. Integrated gradients, and the completeness check

For each sequence model and seed: take each engine's last window, compute IG, and convert it to cycles with the model's target scale. Two checks:

- the network's output at the last window must equal the saved #71 test prediction;
- the attributions must add up to that prediction minus the baseline prediction, up to the error of 256 integration steps.

In [2]:
SEQUENCE = {"transformer": "Transformer", "lstm": "LSTM", "tcn": "Temporal CNN"}
ig, checks = {}, []
for key in SEQUENCE:
    for seed in SEEDS:
        trained = load(key, seed)
        (x,) = last_windows(trained)
        net = trained.model.eval()
        with torch.no_grad():
            pred = net(x).numpy().astype(float) * trained.target_scale
            base = net(torch.zeros_like(x)).numpy().astype(float) * trained.target_scale
        attr = integrated_gradients(net, x).detach().numpy().astype(float) * trained.target_scale  # [engines, window, sensors]
        ig[key, seed] = attr
        saved = final.query("model == @key and target == 'cap125' and seed == @seed").sort_values("unit")["prediction"]
        gap = attr.sum(axis=(1, 2)) - (pred - base)
        checks.append({"model": SEQUENCE[key], "seed": seed,
                       "max |prediction - #71 prediction|": np.abs(pred - saved.to_numpy()).max(),
                       "baseline prediction": base.mean(),
                       "max |completeness gap|": np.abs(gap).max(),
                       "mean |prediction - baseline|": np.abs(pred - base).mean()})
SENSORS = trained.inputs.preprocessor_.feature_columns_
checks = pd.DataFrame(checks)
display(checks.round(4))
assert checks["max |prediction - #71 prediction|"].max() < 1e-3, "checkpoints do not reproduce the #71 predictions"

,model,seed,max |prediction - #71 prediction|,baseline prediction,max |completeness gap|,mean |prediction - baseline|
0,Transformer,42,0.0,126.7808,0.0010,46.8692
1,Transformer,43,0.0,131.1619,0.0006,53.3616
2,Transformer,44,0.0,133.3903,0.0009,54.4902
3,Transformer,45,0.0,127.8542,0.0006,49.0610
4,Transformer,46,0.0,127.1253,0.0002,50.8445
5,LSTM,42,0.0,115.3079,0.0015,39.1088
6,LSTM,43,0.0,114.9005,0.0008,38.2335
7,LSTM,44,0.0,119.6961,0.0039,42.9285
8,LSTM,45,0.0,118.2074,0.0016,40.9637
9,LSTM,46,0.0,113.2151,0.0011,38.0930
